In [1]:
print('start kernel')

start kernel


In [3]:
from pathlib import Path

root = Path.cwd()

In [29]:
import openai 
import langchain
import pinecone
from langchain_community.document_loaders import PyPDFDirectoryLoader,csv_loader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import pinecone
from langchain_openai import OpenAI, ChatOpenAI

In [6]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [7]:
def read_doc(dir):
    file_loader = PyPDFDirectoryLoader(dir)
    
    docs = file_loader.load()

    return docs

docs = read_doc('data/')


In [8]:
len(docs)

58

In [9]:
def divide_chunks(docs,chunk_size=800,chunk_overlap=80):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size- chunk_size,
        chunk_overlap=chunk_overlap,
    )
    docs = text_splitter.split_documents(docs)

    return docs

In [10]:
docs = divide_chunks(docs=docs)
len(docs)

56

In [11]:
embedding_model = OpenAIEmbeddings(api_key=os.environ['OPENAI_API_KEY'],
                                   model='text-embedding-3-small')

In [12]:
vectors = embedding_model.embed_query('hi how are you ?? ')
vectors

[0.0035149615723639727,
 -0.042120907455682755,
 -0.019066981971263885,
 0.046741075813770294,
 0.0012327284784987569,
 -0.026829799637198448,
 0.03445190191268921,
 0.0045351507142186165,
 -0.05037623271346092,
 -0.03414701670408249,
 -0.0009740166715346277,
 -0.020767295733094215,
 -0.03330272436141968,
 -0.01857447624206543,
 0.016440287232398987,
 0.0282135047018528,
 -0.019700203090906143,
 -0.01060058455914259,
 0.019371865317225456,
 0.0255868099629879,
 0.03395939618349075,
 -0.011339342221617699,
 0.002974085509777069,
 -0.00560517655685544,
 0.0063849762082099915,
 0.01842203363776207,
 0.02110735885798931,
 0.009117206558585167,
 0.01240643672645092,
 0.0035296196583658457,
 0.023393990471959114,
 -0.02389822155237198,
 0.006091818679124117,
 0.002789396094158292,
 0.009955638088285923,
 -0.007106144446879625,
 -0.0026193647645413876,
 0.032622598111629486,
 -0.022690411657094955,
 -0.053987935185432434,
 -0.03492095321416855,
 -0.02619657851755619,
 0.030629124492406845,
 0

In [13]:
len(vectors)

1536

# vector search DB in pinecone

In [14]:
import pinecone
print(dir(pinecone))
import pinecone
print(pinecone.__file__)

['Admin', 'AwsRegion', 'AzureRegion', 'BackupList', 'BackupModel', 'ByocSpec', 'CloudProvider', 'CollectionDescription', 'CollectionList', 'Config', 'ConfigBuilder', 'ConfigureIndexEmbed', 'CreateIndexForModelEmbedTypedDict', 'DeleteRequest', 'DeletionProtection', 'DescribeIndexStatsRequest', 'DescribeIndexStatsResponse', 'EmbedModel', 'EmbeddingsList', 'FetchByMetadataResponse', 'FetchResponse', 'FilterBuilder', 'ForbiddenException', 'GcpRegion', 'ImportErrorMode', 'IndexEmbed', 'IndexList', 'IndexModel', 'ListConversionException', 'MetadataSchemaFieldConfig', 'Metric', 'ModelInfo', 'ModelInfoList', 'NamespaceDescription', 'NotFoundException', 'Pinecone', 'PineconeApiAttributeError', 'PineconeApiException', 'PineconeApiKeyError', 'PineconeApiTypeError', 'PineconeApiValueError', 'PineconeAsyncio', 'PineconeConfig', 'PineconeConfigurationError', 'PineconeException', 'PineconeProtocolError', 'PodIndexEnvironment', 'PodSpec', 'PodSpecDefinition', 'PodType', 'QueryRequest', 'QueryResponse'

In [15]:
from pinecone import Pinecone
import os

pc = Pinecone(
    api_key=os.environ["PINECONE_API_KEY"]
)

index = pc.Index(
    host=os.environ["PINECONE_HOST"]
)

In [16]:
print(index.describe_index_stats())

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '151',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 08 Mar 2026 14:43:05 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '52',
                                    'x-pinecone-request-latency-ms': '51',
                                    'x-pinecone-response-duration-ms': '53'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


# Insert Documents into Pinecone

In [ ]:
from tqdm.auto import tqdm # show progress bar

# Prepare documents for insertion
batch_size = 100   #insert 100 vectors in 1 api call 
vectors_to_upsert = [] # store vecotr,metadat before push to pinecone

print(f"Processing {len(docs)} document chunks...")

for i, doc in enumerate(tqdm(docs)): # loop over the chunks

    # Generate embedding for the document
    embedding = embedding_model.embed_query(doc.page_content)
    
    # Prepare metadata
    metadata = {
        'text': doc.page_content,
        'source': doc.metadata.get('source', ''),
        'page': doc.metadata.get('page', 0)
    }
    
    # Create vector tuple (id, embedding, metadata) for each vector.
    vector_id = f"doc_{i}"
    vectors_to_upsert.append((vector_id, 
                              embedding, 
                              metadata))
    
    # Upsert in batches
    if len(vectors_to_upsert) >= batch_size: # if 100 vector is ready then push to pinecone.
        index.upsert(vectors=vectors_to_upsert)
        vectors_to_upsert = [] # after upload empy the list . 

# শেষ batch যদি 100 এর কম হয়,
if vectors_to_upsert:
    index.upsert(vectors=vectors_to_upsert)
    
print("✓ All documents embedded and stored in Pinecone!")
print(index.describe_index_stats())

Processing 56 document chunks...


100%|██████████| 56/56 [00:15<00:00,  3.72it/s]


✓ All documents embedded and stored in Pinecone!
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '184',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 08 Mar 2026 14:43:25 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '40',
                                    'x-pinecone-request-latency-ms': '40',
                                    'x-pinecone-response-duration-ms': '42'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 56}},
 'storageFullness': 0.0,
 'total_vector_count': 56,
 'vector_type': 'dense'}


In [18]:
## consine similarity retreive 

def retrieve_query(query, k=5): 
    """
    Retrieve top-k most similar documents from Pinecone based on cosine similarity.
    
    Args:
        query (str): The input query text
        k (int): Number of top similar results to return (default: 5)
    
    Returns:
        list: List of matching results with scores, text, and metadata
    """
    # Embed the query
    query_embedding = embedding_model.embed_query(query)
    
    # Query Pinecone index
    results = index.query(
        vector=query_embedding,
        top_k=k,
        include_metadata=True
    )
    
    # Format the results
    matching_results = []
    for match in results['matches']:
        result = {
            'id': match['id'],
            'score': match['score'],
            'text': match['metadata'].get('text', ''),
            'source': match['metadata'].get('source', ''),
            'page': match['metadata'].get('page', 0)
        }
        matching_results.append(result)
    
    return matching_results

In [19]:
# Test the retrieve_query function
test_query = "tell me about indian budget?"
results = retrieve_query(test_query, k=3)

# Display results
print(f"Query: {test_query}\n")
print(f"Top {len(results)} matching results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. Score: {result['score']:.4f}")
    print(f"   Source: {result['source']}, Page: {result['page']}")
    print(f"   Text: {result['text'][:200]}...")
    print()

Query: tell me about indian budget?

Top 3 matching results:

1. Score: 0.5723
   Source: data\data.pdf, Page: 4
   Text: Budget 2023-2024 
 
Speech of 
Nirmala Sitharaman 
Minister of Finance 
February 1, 2023 
Hon’ble Speaker,  
 I present the Budget for 2023-24. This is the first Budget in Amrit 
Kaal. 
Introduction 
...

2. Score: 0.5192
   Source: data\data.pdf, Page: 28
   Text: 25 
 
 
 
expenditure. Parts of the outlay will also be linked to, or allocated for, the 
following purposes: 
 Scrapping old government vehicles, 
 Urban planning reforms and actions, 
 Financing ...

3. Score: 0.5160
   Source: data\data.pdf, Page: 8
   Text: 5 
 
 
 
4) Green Growth: We are implementing many programmes for green 
fuel, green energy, green farming, green mobility, green buildings, 
and green equipment, and policies for efficient use of ene...



In [30]:
# Initialize LLM for generating responses
LLM_model = ChatOpenAI(
    api_key=os.environ['OPENAI_API_KEY'],
    model='gpt-4o-mini',  # Using ChatGPT model
    temperature=0.7,
    max_tokens=500
)

In [ ]:
def create_prompt(query, context_docs):
    """
    Create a prompt with context from retrieved documents.
    
    Args:
        query (str): User's question
        context_docs (list): List of retrieved documents with text and metadata
    
    Returns:
        str: Formatted prompt with context
    """
    context = "\n\n".join([f"Context {i+1}:\n{doc['text']}" for i, doc in enumerate(context_docs)])
    
    prompt = f"""You are a helpful AI assistant. Answer the question based on the following context.
    If you cannot answer the question based on the context, say "I don't have enough information to answer this question."

    Context:
    {context}

    Question: {query}

    Answer:"""
        
    return prompt


def rag_query(query, k=3, verbose=True):
    """
    Main RAG function: Retrieve relevant documents and generate an answer.
    
    Args:
        query (str): User's question
        k (int): Number of documents to retrieve (default: 3)
        verbose (bool): Print intermediate steps (default: True)
    
    Returns:
        dict: Contains 'answer', 'sources', and 'retrieved_docs'
    """
    # Step 1: Retrieve relevant documents
    if verbose:
        print("🔍 Searching for relevant documents...")
    
    retrieved_docs = retrieve_query(query, k=k)
    
    if verbose:
        print(f"✓ Retrieved {len(retrieved_docs)} documents\n")
    
    # Step 2: Create prompt with context
    prompt = create_prompt(query, retrieved_docs)
    
    if verbose:
        print(" Generated prompt with context\n")
    
    # Step 3: Get LLM response
    if verbose:
        print(" Generating answer...\n")
    
    answer = LLM_model.invoke(prompt)
    
    # Step 4: Format sources
    sources = [
        {
            'source': doc['source'],
            'page': doc['page'],
            'score': doc['score']
        }
        for doc in retrieved_docs
    ]
    
    # Display results
    if verbose:
        print("="*80)
        print(f"Question: {query}")
        print("="*80)
        print(f"\n📌 Answer:\n{answer}\n")
        print("-"*80)
        print("\n📚 Sources:")
        for i, source in enumerate(sources, 1):
            print(f"  {i}. {source['source']} (Page {source['page']}, Score: {source['score']:.4f})")
        print("="*80)
    
    return {
        'answer': answer,
        'sources': sources,
        'retrieved_docs': retrieved_docs
    }

In [31]:
# Test the RAG query function
response = rag_query("tell me about indian budget?", k=3, verbose=True)

🔍 Searching for relevant documents...
✓ Retrieved 3 documents

 Generated prompt with context

 Generating answer...

Question: tell me about indian budget?

📌 Answer:
content="The Indian Budget for 2023-24, presented by Finance Minister Nirmala Sitharaman on February 1, 2023, aims to build on the foundation laid in the previous budget and envisions a prosperous and inclusive India. Key highlights include:\n\n1. **Economic Growth**: India's economic growth is estimated to be 7% for the current year, the highest among major economies, despite global challenges like the COVID-19 pandemic and geopolitical tensions.\n\n2. **Resilience and Reforms**: The budget emphasizes the importance of wide-ranging reforms and sound policies, which helped India navigate multiple crises effectively.\n\n3. **Total Receipts and Expenditure**: For 2023-24, total receipts (excluding borrowings) are estimated at ₹27.2 lakh crore, while total expenditure is projected at ₹45 lakh crore, with net tax receipts es

In [ ]:
# You can also get just the answer without verbose output
result = rag_query("What are the key points?", k=2, verbose=False)
print(result['answer'])